# 7.19 — Instance & panoptic segmentation

Instance and panoptic segmentation add identity to pixels: not just what class is here, but which object owns this region.

Semantic segmentation supplies the per-pixel class label, while R-CNN style regions provide object handles. Mask R-CNN joins them with mask IoU and panoptic conflict resolution so duplicate, missing, or overlapping instances become measurable.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

## The concept, built once: masks need owners

The core mask metric is intersection over union,

$$\operatorname{IoU}(A,B)=rac{|A\cap B|}{|A\cup B|}.$$

A semantic class map can say four pixels are class 1, but instance segmentation must also say which object owns each pixel.

In [ ]:
def mask_iou(a, b):
    a = np.asarray(a, dtype=bool)
    b = np.asarray(b, dtype=bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union

mask_a = np.array([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
])
mask_b = np.array([
    [1, 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
])
example_iou = mask_iou(mask_a, mask_b)
assert example_iou == 0.75
print("mask IoU", example_iou)

## RoIAlign and panoptic ownership

RoIAlign keeps fractional coordinates. Interpolating halfway between values 10 and 14 gives 12, while rounding to the left cell gives 10. Panoptic output then resolves overlaps so each pixel has exactly one owner.

In [ ]:
def roi_align_1d(values, x):
    left = int(np.floor(x))
    right = min(left + 1, len(values) - 1)
    alpha = x - left
    return (1 - alpha) * values[left] + alpha * values[right]

def instance_segment(instance_masks):
    owner = np.zeros_like(instance_masks[0], dtype=int)
    for idx, mask in enumerate(instance_masks, start=1):
        free = np.logical_and(mask.astype(bool), owner == 0)
        owner[free] = idx
    semantic = owner > 0
    return semantic, owner

roi_values = np.array([10, 14])
aligned = roi_align_1d(roi_values, 0.5)
rounded = roi_values[int(round(0.5))]
inst1 = np.zeros((4, 4), dtype=int)
inst2 = np.zeros((4, 4), dtype=int)
inst1[0:2, 0:2] = 1
inst2[2:4, 2:4] = 1
semantic, panoptic = instance_segment([inst1, inst2])
background = int((panoptic == 0).sum())
assert aligned == 12
assert rounded == 10
assert int(inst1.sum()) == 4
assert int(inst2.sum()) == 4
assert background == 8
print("RoIAlign", aligned, "rounding", rounded, "background", background)

## Synthetic scene ladder

D1 is the hand-built two-object patch. D5 adds more objects, occlusion, and a deliberately imperfect proposal so the same mask IoU calculation faces a harder scene.

In [ ]:
def rectangle_mask(size, top, left, height, width):
    mask = np.zeros((size, size), dtype=int)
    mask[top:top + height, left:left + width] = 1
    return mask

def make_scene(size, specs, miss_last=False):
    gt = []
    pred = []
    for top, left, height, width in specs:
        mask = rectangle_mask(size, top, left, height, width)
        gt.append(mask)
        proposal = mask.copy()
        if miss_last:
            ys, xs = np.where(proposal == 1)
            proposal[ys[-1], xs[-1]] = 0
        pred.append(proposal)
    return gt, pred

scene_specs = [
    ("D1 two boxes", 4, [(0, 0, 2, 2), (2, 2, 2, 2)], False),
    ("D2 three boxes", 6, [(0, 0, 2, 2), (0, 3, 2, 2), (3, 1, 2, 3)], True),
    ("D3 touching objects", 8, [(1, 1, 3, 2), (1, 3, 3, 2), (5, 2, 2, 3)], True),
    ("D4 occluded objects", 10, [(1, 1, 3, 3), (2, 3, 3, 3), (6, 2, 3, 2), (6, 6, 2, 3)], True),
    ("D5 cluttered panoptic", 12, [(1, 1, 3, 3), (1, 5, 3, 3), (5, 2, 4, 2), (6, 6, 3, 3), (8, 0, 2, 3)], True),
]
scenes = []
for name, size, specs, miss_last in scene_specs:
    gt, pred = make_scene(size, specs, miss_last=miss_last)
    scenes.append((name, gt, pred))

fig, axes = plt.subplots(1, 5, figsize=(10, 2.2))
for ax, (name, gt, pred) in zip(axes, scenes):
    semantic, owners = instance_segment(gt)
    ax.imshow(owners, cmap="tab20", vmin=0)
    ax.set_title(name, fontsize=8)
    ax.axis("off")
plt.tight_layout()

In [ ]:
iou_scores = []
for name, gt, pred in scenes:
    per_instance = [mask_iou(a, b) for a, b in zip(gt, pred)]
    score = float(np.mean(per_instance))
    iou_scores.append(score)
    print(f"{name:24s} IoU={score:.3f} instances={len(gt)}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4.2))
for col, (name, gt, pred) in enumerate(scenes):
    _, gt_owner = instance_segment(gt)
    _, pred_owner = instance_segment(pred)
    axes[0, col].imshow(gt_owner, cmap="tab20", vmin=0)
    axes[0, col].set_title(name, fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(pred_owner, cmap="tab20", vmin=0)
    axes[1, col].set_title(f"pred IoU {iou_scores[col]:.2f}", fontsize=8)
    axes[1, col].axis("off")
plt.tight_layout()

plt.figure(figsize=(5, 3))
plt.plot(range(1, 6), iou_scores, marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("mean mask IoU")
plt.title("Instance mask quality across the ladder")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on D5: semantic masks are not instance masks

Merging same-class objects can keep a plausible class-1 semantic region while destroying object identity. The fix is to keep instance IDs and resolve overlap to one panoptic owner per pixel.

In [ ]:
name, gt, pred = scenes[-1]
semantic_union = np.logical_or.reduce([m.astype(bool) for m in gt])
merged_prediction = semantic_union.astype(int)
wrong_count = int(merged_prediction.max())
_, fixed_owner = instance_segment(gt)
fixed_count = len(set(fixed_owner.ravel().tolist()) - {0})
overlap = gt[0].astype(bool)
overlap[1:4, 5:8] = True
_, resolved_owner = instance_segment([overlap.astype(int), gt[1]])
unique_pixels = int(np.isin(resolved_owner, [1, 2]).sum())
print("wrong semantic object count", wrong_count)
print("fixed instance object count", fixed_count)
print("resolved owned pixels", unique_pixels)
assert wrong_count == 1
assert fixed_count == 5

## Evaluate it + Practice

- Metric: mean mask IoU; compare with the no-skill baseline, one merged semantic blob.
- Cheap sanity check: overfit D1 with perfect proposal masks.
- Ablation: merge IDs and IoU no longer checks object ownership.
- Failure signals: same-class objects collapse or panoptic owners overlap.

Practice:
1. Change one D1 number and predict the metric before running it.

In [ ]:
# Your experiment here

2. Add one harder case to the ladder and rerun the curve.

In [ ]:
# Your harder case here

3. Turn off the key fix in the pitfall cell and explain the change.

In [ ]:
# Your ablation here